# 모델 배포 개론 08 — [자율 프로젝트] 청년 대상 금융사기·불법 리딩방 위험 탐지기
**Youth Financial Scam Guard (FastAPI & Streamlit Multimodal FDS)**  
Last modified : 2026.08  
작성 : 박광성 (모두의연구소)  
수정 : 김지성, 박기웅 (모두의연구소)  
진행 : 천세문 (엔지니어 3기)

---

> ### **🎯 프로젝트 목표 (Project Objective)**
>
> 2026년 고금리·자산 격차 심화 속에서 2030 청년층(사회초년생, 대학생, 취준생)을 집중 타깃으로 급증하는 **5대 지능형 금융사기(불법 리딩방, SNS 선입금 알바, 폰테크/대리입금, 정책금융·대환대출 사칭, 기관 사칭 피싱)**를 사전 차단하는 **엔드투엔드(End-to-End) 멀티모달 금융 안전망 서빙 시스템**을 구축합니다.
>
> 1. **멀티모달 Vision OCR 파이프라인 탑재**:  
>    텍스트 직접 입력뿐만 아니라, 청년들이 일상에서 수신하는 **카카오톡·문자·텔레그램 스크린샷 캡처 사진에서 EasyOCR을 통해 텍스트를 실시간으로 자동 추출**합니다.
> 2. **실제 금융권 FDS(이상거래탐지) 4대 차원 하이브리드 스코어링 (0~100점)**:  
>    단순 텍스트 분류를 넘어 `[D1 시그니처 룰(40점)]` + `[D2 위협 인텔리전스 URL(25점)]` + `[D3 KR-FinBert-SC 감정 및 심리 조작(20점)]` + `[D4 수신 채널 위험도(15점)]`의 4대 차원 정밀 분해 점수를 산출합니다.
> 3. **프로덕션급 고성능 백엔드 & 대시보드 구축**:  
>    **FastAPI 게이트웨이 + Pydantic V2 데이터 유효성 검증 + 비동기 스레드 풀(`run_in_executor`) 격리 + `X-API-Key` 헤더 인증 + Streamlit 대시보드 UI**를 결합하여, 0.1초 내 위험 진단 및 **금융감독원(1332) 간편 신고서 자동 생성**을 지원하는 실전 AI 안전망을 구현합니다.

---


In [1]:
# ── 서버 실행 도우미 (노트북 맨 처음에 한 번 실행) ──────────────────
# 노트북 안에서 FastAPI (Uvicorn) 및 Streamlit 서버를 백그라운드로 띄우고 멈추는 도우미 함수
import os, sys, asyncio, threading, time, socket, contextlib, importlib, subprocess, tempfile
import uvicorn

# 작업 디렉터리를 프로젝트 루트로 설정
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# 프로젝트 폴더 생성
for _d in ('app', 'models', 'data', 'frontend', 'images'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 Uvicorn 서버를 안전하게 종료합니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드 스레드에서 Uvicorn 서버를 실행합니다."""
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다.')
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    for i in range(600):
        if _port_open(host, port):
            print(f'✅ 서버 실행 완료: http://{host}:{port}')
            return server
        if not thread.is_alive():
            print('❌ 서버 스레드가 비정상 종료되었습니다.')
            return server
        time.sleep(0.5)
    return server

def run_streamlit(script="frontend/app.py", port=8501):
    """Streamlit을 백그라운드 프로세스로 실행합니다."""
    if _port_open("127.0.0.1", port):
        print(f"♻️  Streamlit이 이미 실행 중입니다: http://127.0.0.1:{port}")
        return None
    log_path = os.path.join(tempfile.gettempdir(), f"streamlit_{port}.log")
    log = open(log_path, "w", encoding="utf-8")
    proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", script,
         "--server.port", str(port),
         "--server.enableCORS", "false",
         "--server.enableXsrfProtection", "false",
         "--server.headless", "true"],
        stdout=log, stderr=subprocess.STDOUT
    )
    for _ in range(60):
        if proc.poll() is not None:
            log.close()
            print(f"❌ Streamlit 시작 실패 — 로그 확인: {log_path}")
            return proc
        if _port_open("127.0.0.1", port):
            print(f"✅ Streamlit 프론트엔드 실행 완료: http://127.0.0.1:{port}")
            return proc
        time.sleep(0.25)
    return proc

print('🚀 서버 도우미 준비 완료 (serve_in_thread, run_streamlit, stop_server)')

🚀 서버 도우미 준비 완료 (serve_in_thread, run_streamlit, stop_server)


---

## 1장. 프로젝트 개요 및 서비스 기획 (Intro)

### 1.1 사회초년생·청년 대상 금융사기 문제 정의
- **사회적 배경**: 최근 2030 청년을 타깃으로 한 **불법 주식/코인 리딩방, SNS 고수익 단기 알바(선입금 피싱), 폰테크/소액결제 현금화, 정부지원 저금리 대환대출 사칭** 등 지능형 금융사기가 급증하고 있습니다.
- **기존 방식의 한계**: 통신사 스팸 필터는 단순 키워드 차단에 그치며, 일반 사용자가 수신한 문자/카톡의 법적 위험도와 사기 여부를 다차원으로 즉시 판별해 주는 전문 서비스가 부재합니다.
- **해결책 (본 프로젝트)**: 금융 특화 언어 모델(`snunlp/KR-FinBert-SC`)과 **실제 금융권 FDS 4대 차원 스코어링 엔진** 및 **EasyOCR 멀티모달 비전 파이프라인**을 결합하여, 실시간으로 정밀 위험 점수(0~100점)와 **금융감독원(1332) 원클릭 신고 서식**을 제공하는 상용급 안전망을 구축합니다.

> ### 📸 [스크린샷 1] 청년 타깃 지능형 금융사기 실제 사례 스크린샷
>
> ![실제 청년 금융사기 피해 사례](images/01_scam_samples.png)
>
> * **출처**: [잡플래닛 컴퍼니타임스 — "두시간 일하고 월 300 누구나 가능? 이런건 없어요"](https://www.jobplanet.co.kr/contents/news-5718/%EB%91%90%EC%8B%9C%EA%B0%84-%EC%9D%BC%ED%95%98%EA%B3%A0-%EC%9B%94-300-%EB%88%84%EA%B5%AC%EB%82%98-%EA%B0%80%EB%8A%A5-%EC%9D%B4%EB%9F%B0%EA%B1%B4-%EC%97%86%EC%96%B4%EC%9A%94)


In [2]:
# ── [Step 1] 사전학습 모델 동작 검증 ────────────────────────────────
from transformers import pipeline

classifier = pipeline("text-classification", model="snunlp/KR-FinBert-SC")

sample_scam = "[긴급] 이번 주 500% 폭등 확정 VIP 리딩방 선착순 10명 무료 입장! 원금 100% 보장!"
res = classifier(sample_scam)
print(f"테스트 문장: {sample_scam}")
print(f"모델 추론 결과: {res}")

C:\Users\cheon\OneDrive\work\model-serving-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15023.17it/s]


테스트 문장: [긴급] 이번 주 500% 폭등 확정 VIP 리딩방 선착순 10명 무료 입장! 원금 100% 보장!
모델 추론 결과: [{'label': 'neutral', 'score': 0.9997124075889587}]


### 1.3 사전학습 모델의 한계 분석 및 하이브리드 FDS 설계 당위성

> ### **[추론 결과 해석: AI 모델은 왜 사기 문자를 '중립(99.9%)'으로 오판했는가?]**
>
> 사람에게는 100% 명백한 불법 사기 문자(`"500% 폭등 확정"`, `"원금 100% 보장"`)임에도 불구하고, `KR-FinBert-SC`가 **`neutral (중립, 99.97%)`**이라는 엉뚱한 예측을 내놓은 데에는 **3가지 기술적 한계**가 존재합니다:
>
> 1. **학습 데이터 도메인의 근본적 괴리 (Domain Gap)**:  
>    `KR-FinBert-SC`는 정제된 **네이버 금융 뉴스 기사 및 기업 실적 공시(DART) 텍스트**로 학습되었습니다. 모델의 학습 데이터셋에는 불특정 다수에게 살포되는 비정형 스미싱이나 허위 과장 광고 코퍼스가 전혀 존재하지 않습니다.
> 2. **태스크 목적의 불일치 (Task Mismatch)**:  
>    이 모델의 분류 목적은 "해당 문장이 기업 주가에 호재(`Positive`)인가, 악재(`Negative`)인가, 단순 사실 전달(`Neutral`)인가"를 구분하는 것입니다. 모델의 시각에서 `"선착순 10명 무료 입장"`, `"정보방 링크"`는 기업 가치 상승/하락과 무관한 '단순 안내성 일반 문장(`Neutral`)'으로 기계적으로 처리된 것입니다.
> 3. **금융 법률 및 사기 범죄 지식의 부재**:  
>    모델 가중치(Weights)에는 "원금을 보장하며 투자금을 모집하는 행위는 유사수신행위규제법 및 자본시장법 위반 범죄"라는 법률적·사회적 개념이 전혀 임베딩되어 있지 않습니다.
>
> ---
>
> 🚨 **엔지니어링 결론 및 하이브리드 FDS 아키텍처 설계 당위성**:  
> * 사전학습된 AI 언어 모델 하나에만 의존하여 서비스를 서빙할 경우, 가장 치명적인 사기 범죄 문자가 '안전(중립)'으로 통과되는 치명적인 보안 홀(False Negative)이 발생합니다.  
> * 따라서 법적 불법 시그니처와 피싱 링크는 [D1 시그니처 룰(40점)] + [D2 위협 링크(25점)]로 100% 확실하게 차단하고, AI 모델은 심리적 과장 어조를 보조 측정하는 [D3 심리 의도(20점)] + [D4 수신 채널(15점)]의 **실제 시중은행 FDS 결합형 4대 차원 하이브리드 서빙 아키텍처**를 구축하게 된 결정적 근거가 됩니다.


---

## 2장. 시스템 아키텍처 및 파이프라인 설계

### 2.1 계층별 시스템 구조 (엔터프라이즈 FDS & 멀티모달 아키텍처)

```text
┌──────────────────────────────────────────────────────────┐
│          [1. Client Layer: Streamlit UI (포트 8501)]      │
│  • 탭 1: ✍️ 텍스트 직접 입력 모드                          │
│  • 탭 2: 📸 카톡/문자 캡처 사진 업로드 (OCR)              │
└────────────────────────────┬─────────────────────────────┘
                             │ (POST /predict or /predict/image)
                             │ [Header: X-API-Key]
                             ▼
┌──────────────────────────────────────────────────────────┐
│        [2. Gateway Layer: FastAPI Server (포트 8000)]    │
│  • 보안 인증: app/auth.py (401 비인가 차단)              │
│  • 데이터 검증: app/schemas.py (Pydantic 422 방어)        │
│  • 비동기 워커 풀: ThreadPoolExecutor 격리 실행          │
└────────────────────────────┬─────────────────────────────┘
                             │
            ┌────────────────┴────────────────┐
            ▼ [이미지 업로드 분기]             ▼ [텍스트 입력 분기]
┌───────────────────────────┐                 │
│ [3. Vision OCR Layer]     │                 │
│   (app/ocr_service.py)    │                 │
│ • EasyOCR 한글/영문 추출  │                 │
│ • 이미지 노이즈 정제      │ ──(추출 텍스트)─┘
└───────────┬───────────────┘
            │
            ▼
┌──────────────────────────────────────────────────────────┐
│      [4. Engine Layer: 지능형 금융 FDS 스코어링]          │
│                 (app/model_service.py)                   │
│ ──────────────────────────────────────────────────────── │
│  • [D1] 시그니처 룰 (40점) : 5대 사기 핵심/보조 키워드   │
│  • [D2] 위협 인텔리전스 (25점): 피싱/단축 URL 추적       │
│  • [D3] AI 심리 의도 (20점) : 긴급성 선동 + KR-FinBERT  │
│  • [D4] 수신 채널 (15점)   : Telegram / DM 위험 가중치  │
└────────────────────────────┬─────────────────────────────┘
                             │
                             ▼
┌──────────────────────────────────────────────────────────┐
│           [5. Final Output: 최종 분석 결과]              │
│  • FDS 종합 위험 점수 (0~100점) & 3단계 위험 등급        │
│  • 5대 사기 유형별 위험 기여도 분포                      │
│  • 금융감독원 1332 원클릭 간편 제보 서식 자동 완성       │
└──────────────────────────────────────────────────────────┘
```

#### 📌 계층별 상세 역할 및 엔지니어링 설계 의도
1. **프론트엔드 표현 계층 (Presentation Layer)**:
   - Streamlit 기반으로 텍스트 및 스크린샷 캡처 이미지의 멀티모달 입력을 처리하고, FDS 4대 차원 분석 결과와 금감원 1332 신고 서식을 시각화합니다.
2. **보안 & 게이트웨이 계층 (API Gateway & Security Layer)**:
   - FastAPI 및 `X-API-Key` 기반 의존성 주입(`Depends(verify_api_key)`)을 통해 비인가 요청을 컨트롤러 진입 전 401로 차단합니다.
3. **멀티모달 비전 계층 (Vision OCR Layer)**:
   - 사용자가 올린 휴대폰 화면 캡처 사진에서 EasyOCR을 통해 비동기로 텍스트를 추출하여 텍스트 파이프라인과 통합합니다.
4. **지능형 FDS 추론 계층 (Inference & Scoring Layer)**:
   - `snunlp/KR-FinBert-SC`의 딥러닝 감정 분석과 실제 시중은행 FDS 4대 룰베이스를 결합하여 0~100점의 연속 위험 점수를 산출합니다.

### 2.2 프로젝트 디렉터리 및 모듈 구조

```text
model-serving-course/
├── 📁 app/                               # FastAPI 백엔드 핵심 애플리케이션 모듈
│   ├── 📄 __init__.py
│   ├── 🔒 auth.py                        # X-API-Key 헤더 기반 보안 인증 계층
│   ├── 📋 schemas.py                     # Pydantic 기반 FDS 입·출력 데이터 검증 스키마
│   ├── 🧠 model_service.py               # KR-FinBert-SC + 금융 FDS 4대 차원 스코어링 엔진
│   ├── 👁️ ocr_service.py                 # EasyOCR 기반 스크린샷 이미지 텍스트 추출 모듈
│   └── 🚀 main.py                        # FastAPI 애플리케이션 진입점 및 비동기 엔드포인트
│
├── 📁 frontend/                          # 인터랙티브 프론트엔드 대시보드
│   └── 🖥️ app.py                         # Streamlit 멀티모달(텍스트+OCR) FDS 대시보드
│
├── 📁 data/                              # 테스트용 데이터셋 및 미디어 파일
│   └── 🖼️ sample_scam_screenshot.png     # 멀티모달 OCR 검증용 사기 문자 캡처 이미지
│
├── 📁 images/                            # 보고서 및 문서화용 스크린샷 갤러리
│   ├── 🖼️ 01_scam_samples.png
│   ├── 🖼️ 02_streamlit_text_demo.png
│   ├── 🖼️ 03_streamlit_ocr_demo.png
│   └── 🖼️ 04_fastapi_swagger_docs.png
│
└── 📁 notebooks/                         # 프로젝트 단계별 파이프라인 노트북
    └── 📓 모델배포개론08_자율프로젝트.ipynb # 전체 엔드투엔드 실습 및 검증 주피터 노트북
```

---

## 3장. 보안 인증 및 Pydantic 데이터 검증 계층 구축

### 3.1 API Key 헤더 인증 모듈 (`app/auth.py`)
- `X-API-Key` 헤더를 검증하여 인가된 클라이언트만 추론 엔드포인트에 접근할 수 있도록 차단(`401 Unauthorized`)합니다.

In [3]:
%%writefile app/auth.py
"""
보안 및 API Key 인증 모듈
"""
from fastapi import HTTPException, Header

VALID_API_KEYS = {
    "test-key-001": "청년보호센터_관리자",
    "test-key-002": "금융안전_사용자",
}

async def verify_api_key(x_api_key: str = Header(None)) -> str:
    """X-API-Key 헤더를 검증하여 사용자명을 반환하거나 401 예외를 발생시킵니다."""
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )
    return VALID_API_KEYS[x_api_key]


Overwriting app/auth.py


### 3.2 Pydantic BaseModel 기반 입·출력 스키마 (`app/schemas.py`)
- 입력 텍스트 길이 제한(5~3000자), 수신 채널 검증, 그리고 FDS 4대 차원(`ScoreBreakdown`) 및 `DetectedSignals`를 엄격한 타입으로 직렬화합니다.

In [4]:
%%writefile app/schemas.py
"""
청년 대상 금융사기·불법 리딩방 위험 탐지기 (Youth Financial Scam Guard) Pydantic 스키마 (v4 금융 FDS 다차원 스코어링)
"""
from typing import List, Dict, Optional, Any
from pydantic import BaseModel, Field


class ScamDetectRequest(BaseModel):
    """금융사기 분석 요청 스키마"""
    text: str = Field(
        ...,
        min_length=5,
        max_length=3000,
        description="분석할 의심 문자, 카카오톡 메시지, SNS DM 내용",
        json_schema_extra={"example": "[WEB 발신] (주)OO커머스 일일 업무 알바 모집. 당일 정산 최소 30만 원 보장, 누구나 즉시 가능! 단시간 고수익 카톡 문의 http://bit.ly/job-scam"}
    )
    channel: str = Field(
        default="SMS/MMS",
        description="메시지 수신 채널 (SMS/MMS, KakaoTalk, Telegram, Instagram DM, OpenChat, 기타)",
        json_schema_extra={"example": "SMS/MMS"}
    )


class ScoreBreakdown(BaseModel):
    """실제 금융권 FDS 4대 차원별 점수 분해 (총 100점)"""
    dimension_1_rule_score: int = Field(..., description="[D1] 시그니처 룰 점수 (최대 40점)")
    dimension_2_threat_score: int = Field(..., description="[D2] 링크/위협 인텔리전스 점수 (최대 25점)")
    dimension_3_intent_score: int = Field(..., description="[D3] AI 언어 & 심리 조작 점수 (최대 20점)")
    dimension_4_channel_score: int = Field(..., description="[D4] 채널 & 컨텍스트 위험 점수 (최대 15점)")


class DetectedSignals(BaseModel):
    """세부 탐지 시그널 딕셔너리"""
    critical_keywords: List[str] = Field(default=[], description="핵심 범죄 키워드")
    minor_keywords: List[str] = Field(default=[], description="보조 의심 키워드")
    urgency_keywords: List[str] = Field(default=[], description="긴급성/FOMO 심리 조작 단어")
    suspicious_urls: List[str] = Field(default=[], description="피싱/단축 의심 링크")
    channel_risk_info: str = Field(..., description="수신 채널 위험도 분석")


class ScamDetectResponse(BaseModel):
    """금융사기 분석 응답 스키마"""
    success: bool = Field(True, description="요청 성공 여부")
    risk_score: int = Field(..., ge=0, le=100, description="최종 위험 점수 (0~100)")
    risk_level: str = Field(..., description="위험 등급 (안전/주의/경고/고위험)")
    scam_type: str = Field(..., description="주요 판별 사기 유형")
    score_breakdown: ScoreBreakdown = Field(..., description="금융 FDS 4대 차원별 점수 분해")
    category_distribution: Dict[str, int] = Field(..., description="5대 사기 카테고리별 위험 기여도 분포")
    detected_signals: DetectedSignals = Field(..., description="세부 탐지 시그널")
    action_guide: str = Field(..., description="피해 예방 및 즉각 대처 행동 요령")
    model_sentiment: str = Field(..., description="AI 모델 판별 금융 감정 (positive/negative/neutral)")
    model_confidence: float = Field(..., description="AI 모델 감정 신뢰도 (0.0~1.0)")
    user: Optional[str] = Field(None, description="인증된 사용자/클라이언트 식별자")
    latency_ms: float = Field(..., description="분석 소요 시간 (밀리초)")


class ImageScamDetectResponse(ScamDetectResponse):
    """이미지 OCR 기반 금융사기 분석 응답 스키마"""
    extracted_text: str = Field(..., description="이미지에서 자동 추출된 텍스트")
    ocr_latency_ms: float = Field(..., description="OCR 텍스트 추출 소요 시간 (밀리초)")


class HealthResponse(BaseModel):
    """시스템 헬스체크 응답 스키마"""
    status: str
    model_loaded: bool
    service_name: str
    version: str


Overwriting app/schemas.py


---

## 4장. 금융 FDS 지능형 추론 엔진 및 FastAPI 백엔드 완성

### 4.1 지능형 금융사기 스코어링 엔진 (`app/model_service.py`)
- `normalize_text` (특수문자/공백 분리 회피 방어: `원.금.보.장`, `폰/테/크`)
- FDS 4대 차원 스코어링: [D1 룰 40점] + [D2 링크 25점] + [D3 심리 20점] + [D4 채널 15점]
- 5대 사기 유형 기여도 분포 및 금감원 1332 행동 요령 생성

In [5]:
%%writefile app/model_service.py
"""
청년 금융사기 탐지(Youth Financial Scam Guard) 엔터프라이즈 FDS 엔진 (v4)
- 실제 금융권 FDS 4대 차원별 점수 분해 (Score Decomposition)
  1. Dimension 1: 시그니처 룰 점수 (Rule & Signature Score, 최대 40점)
  2. Dimension 2: 위협 인텔리전스 점수 (Threat Intelligence URL Score, 최대 25점)
  3. Dimension 3: AI 언어 & 행동 심리 조작 점수 (NLP Behavioral Intent Score, 최대 20점)
  4. Dimension 4: 채널 & 컨텍스트 위험 점수 (Channel & Context Score, 최대 15점)
- 5대 사기 유형별 다차원 위험 기여도 분포 (Multi-Category Probability Distribution)
- 긴급성 / FOMO 조작 탐지기 (Urgency & Scarcity Detector)
- 특수문자 및 자모 분리 회피 방어 (Whitespace & Delimiter Invariant Normalization)
"""
import time
import re
import unicodedata
from typing import Dict, Any, List, Tuple
from transformers import pipeline

MODEL_NAME = "snunlp/KR-FinBert-SC"

# 1. 텍스트 정규화 헬퍼 (특수문자/공백 분리 회피 방어)
def normalize_text(text: str) -> str:
    normalized = unicodedata.normalize('NFKC', text)
    clean_text = re.sub(r"[!@#$%^&*()_+\-=\[\]{};':\"\\|,.<>\/?~`]", " ", normalized)
    clean_text = re.sub(r"\s+", " ", clean_text).strip()
    return clean_text


# 2. 5대 사기 카테고리별 시그니처 룰베이스 (Critical: +20점, Minor: +10점)
SCAM_CATEGORIES_RULES = {
    "불법 주식/코인 리딩방": {
        "critical_patterns": [
            r"(원\s*금\s*보\s*장|원\s*금\s*100\s*%|100\s*%\s*보\s*장)",
            r"(수\s*익\s*률\s*\d+\s*%|\d+\s*%\s*폭\s*등|상\s*한\s*가\s*확\s*정|폭\s*등\s*확\s*정)",
            r"(VIP\s*리\s*딩\s*방|VIP\s*정\s*보\s*방|세\s*력\s*매\s*집\s*주|비\s*상\s*장\s*주\s*식\s*상\s*장)"
        ],
        "minor_patterns": [
            r"(선\s*착\s*순\s*무\s*료|극\s*비\s*정\s*보|단\s*타\s*수\s*익|내\s*부\s*자\s*정\s*보|수\s*익\s*인\s*증)"
        ],
        "guide": "[주의] 원금과 고수익을 동시에 보장하는 주식/코인 리딩방은 100% 불법 유사수신 행위입니다. 링크 접속을 중단하고 단체방을 퇴장하세요."
    },
    "청년 타깃 고수익 SNS 알바": {
        "critical_patterns": [
            r"(당\s*일\s*정\s*산|당\s*일\s*지\s*급|즉\s*시\s*지\s*급)",
            r"(단\s*시\s*간\s*고\s*수\s*익|고\s*수\s*익\s*알\s*바|재\s*택\s*꿀\s*알\s*바|단\s*기\s*알\s*바|재\s*택\s*부\s*업)",
            r"(쇼\s*핑\s*몰\s*(리\s*뷰|알\s*바|부\s*업|홍\s*보|체\s*험\s*단)|상\s*품\s*후\s*기|후\s*기\s*글|리\s*뷰\s*알\s*바|주\s*문\s*서\s*작\s*성)",
            r"(수\s*수\s*료\s*지\s*급|선\s*입\s*금\s*요\s*구|보\s*증\s*금\s*입\s*금|대\s*리\s*구\s*매)"
        ],
        "minor_patterns": [
            r"(간\s*단\s*업\s*무|누\s*구\s*나\s*가\s*능|주\s*부\s*가\s*능|초\s*보\s*가\s*능|장\s*소\s*무\s*관)",
            r"(일\s*당|수\s*당|수\s*익\s*금|월\s*급|수\s*령)\s*(\d+|최\s*소|최\s*대|평\s*균)",
            r"(사\s*대\s*보\s*험\s*미\s*적\s*용|미\s*성\s*년\s*자\s*불\s*가)",
            r"(카\s*톡\s*문\s*의|오\s*픈\s*채\s*팅|라\s*인\s*문\s*의|채\s*널\s*추\s*가)"
        ],
        "guide": "[주의] 쇼핑몰 리뷰, 상품 결제 유도 후 수수료/보증금을 요구하는 재택 알바는 신종 사기입니다. 본인 돈을 먼저 입금하지 마세요."
    },
    "소액 급전 / 폰테크 / 대리입금": {
        "critical_patterns": [
            r"(폰\s*테\s*크|내\s*구\s*제\s*대\s*출|소\s*액\s*결\s*제\s*현\s*금\s*화|비\s*상\s*금\s*현\s*금\s*화)",
            r"(대\s*리\s*입\s*금|개\s*통\s*대\s*납|휴\s*대\s*폰\s*개\s*통\s*대\s*납)",
            r"(통\s*장\s*대\s*여|체\s*크\s*카\s*드\s*양\s*도|명\s*의\s*대\s*여)"
        ],
        "minor_patterns": [
            r"(급\s*전|신\s*불\s*자\s*가\s*능|무\s*직\s*자\s*대\s*출|당\s*일\s*소\s*액|수\s*고\s*비\s*지\s*급|지\s*각\s*비)"
        ],
        "guide": "[위험] 폰테크와 대리입금은 연 1000% 이상의 불법 고금리이며, 대포폰/대포통장 범죄에 연루될 수 있습니다. 서민금융진흥원(1397)을 이용하세요."
    },
    "정부지원 / 저금리 대환대출 사칭": {
        "critical_patterns": [
            r"(국\s*민\s*행\s*복\s*기\s*금|서\s*민\s*금\s*융\s*진\s*흥\s*원|정\s*부\s*지\s*원\s*저\s*금\s*리)",
            r"(저\s*금\s*리\s*대\s*환\s*대\s*출|고\s*금\s*리\s*대\s*환|최\s*대\s*1\.\d+\s*%\s*대\s*환)"
        ],
        "minor_patterns": [
            r"(한\s*도\s*소\s*진\s*전|특\s*별\s*지\s*원\s*대\s*출|무\s*료\s*상\s*담\s*신\s*청|긴\s*급\s*생\s*활\s*안\s*정\s*자\s*금)"
        ],
        "guide": "[위험] 공공기관이나 정부지원 대환대출을 사칭하여 문자 링크로 상담을 유도하는 것은 대출사기 스미싱입니다. 금융사 공식 대표번호로 확인하세요."
    },
    "기관 사칭 / 보이스피싱": {
        "critical_patterns": [
            r"(금\s*융\s*감\s*독\s*원|서\s*울\s*중\s*앙\s*지\s*검|대\s*검\s*찰\s*청)",
            r"(대\s*포\s*통\s*장\s*연\s*루|자\s*산\s*검\s*수|구\s*속\s*영\s*장|계\s*좌\s*동\s*결)",
            r"(안\s*전\s*계\s*좌\s*이\s*체|원\s*격\s*제\s*어\s*앱|팀\s*뷰\s*어|TeamViewer)"
        ],
        "minor_patterns": [
            r"(범\s*죄\s*수\s*사|수\s*사\s*협\s*조|금\s*융\s*보\s*안\s*원)"
        ],
        "guide": "[긴급] 검찰, 경찰, 금감원은 어떠한 경우에도 원격제어 앱 설치나 안전계좌 이체를 요구하지 않습니다. 즉시 전화를 끊고 112 또는 1332로 확인하세요."
    }
}

# 3. 긴급성 & FOMO 심리 조작 패턴 (Urgency Patterns)
URGENCY_PATTERNS = [
    r"(선\s*착\s*순\s*\d+\s*명|선\s*착\s*순\s*마\s*감|오\s*늘\s*마\s*감|금\s*일\s*마\s*감)",
    r"(한\s*도\s*소\s*진\s*임\s*박|한\s*도\s*소\s*진\s*전|조\s*기\s*마\s*감)",
    r"(즉\s*시\s*처\s*리\s*하\s*지\s*않\s*으\s*면|마\s*지막\s*기\s*회|긴\s*급\s*공\s*지)"
]

# 4. 피싱 의심 단축 URL 및 위험 도메인 패턴 (OCR 오인식 및 카톡 채널 방어 포함)
SUSPICIOUS_URL_PATTERNS = [
    r"bit[\.\s\/_\|\-l]+ly/[a-zA-Z0-9_\-]+",
    r"bit[\.\s\/_\|\-l]+ly[a-zA-Z0-9_\-]+",
    r"me2[\.\s\/_\|\-l]+do/[a-zA-Z0-9_\-]+",
    r"t[\.\s\/_\|\-l]+me/[a-zA-Z0-9_\-]+",
    r"pf[\.\s\/_\|\-l]*kakao[\.\s\/_\|\-l]*com[a-zA-Z0-9_\-\/]*",
    r"open[\.\s\/_\|\-l]+kakao[\.\s\/_\|\-l]+com/[a-zA-Z0-9_\-]+",
    r"https?://[^\s]+\.(xyz|top|site|club|vip|link|info|apk)"
]

# 5. 수신 채널별 기본 위험 가중치
CHANNEL_WEIGHTS = {
    "Telegram": {"score": 15, "info": "[위험] 텔레그램은 익명성이 높아 불법 리딩방 및 사기 범죄에 악용되는 대표 채널입니다."},
    "Instagram DM": {"score": 12, "info": "[위험] 인스타그램 DM을 통한 부업/리뷰 알바 제안은 전형적인 선입금 피싱 채널입니다."},
    "OpenChat": {"score": 12, "info": "[위험] 카카오톡 오픈채팅방은 금융사칭 및 불법 리딩방 유인의 주요 통로입니다."},
    "SMS/MMS": {"score": 8, "info": "[주의] [WEB발신] 번호로 유입된 금융/대출 문자는 스미싱 가능성을 확인해야 합니다."},
    "KakaoTalk": {"score": 5, "info": "[주의] 공식 플러스친구 인증 마크가 없는 일반 카톡 계정의 금융 상담은 주의가 필요합니다."},
    "기타": {"score": 5, "info": "[주의] 미등록 비공식 채널을 통한 금융 안내입니다."}
}


def load_model():
    """Hugging Face 사전학습 감정 분석 모델 로드"""
    print(f"[ModelService] 모델 로딩 시작: {MODEL_NAME}")
    model_pipeline = pipeline("text-classification", model=MODEL_NAME)
    print(f"[ModelService] 모델 로딩 완료: {MODEL_NAME}")
    return model_pipeline


def _evaluate_fds_signals(raw_text: str, channel: str) -> Dict[str, Any]:
    """
    실제 금융권 FDS 4대 차원별 점수 계산
    """
    norm_text = normalize_text(raw_text)
    compact_text = re.sub(r"\s+", "", norm_text)

    critical_kws: List[str] = []
    minor_kws: List[str] = []
    urgency_kws: List[str] = []
    suspicious_urls: List[str] = []
    category_scores: Dict[str, int] = {cat: 0 for cat in SCAM_CATEGORIES_RULES}

    # ── [D1] 시그니처 룰 점수 (최대 40점) ───────────────────────────────
    for category, rules in SCAM_CATEGORIES_RULES.items():
        cat_score = 0
        # A. Critical Keywords (+20점/개)
        for pattern in rules["critical_patterns"]:
            m = re.findall(pattern, raw_text, re.IGNORECASE) or re.findall(pattern, norm_text, re.IGNORECASE) or re.findall(pattern, compact_text, re.IGNORECASE)
            if m:
                kw = m[0] if isinstance(m[0], str) else " ".join(m[0])
                critical_kws.append(kw)
                cat_score += 20
        # B. Minor Keywords (+10점/개)
        for pattern in rules["minor_patterns"]:
            m = re.findall(pattern, raw_text, re.IGNORECASE) or re.findall(pattern, norm_text, re.IGNORECASE) or re.findall(pattern, compact_text, re.IGNORECASE)
            if m:
                kw = m[0] if isinstance(m[0], str) else " ".join(m[0])
                minor_kws.append(kw)
                cat_score += 10

        category_scores[category] = cat_score

    # D1 점수: 최대 매칭 카테고리 기준 상한 40점
    d1_rule_score = min(40, max(category_scores.values())) if category_scores else 0

    # ── [D2] 위협 인텔리전스 링크 점수 (최대 25점) ─────────────────────
    for url_pat in SUSPICIOUS_URL_PATTERNS:
        found = re.findall(url_pat, raw_text, re.IGNORECASE)
        if found:
            suspicious_urls.extend(found)

    has_suspicious_url = len(suspicious_urls) > 0
    has_general_url = bool(re.search(r"https?://[^\s]+", raw_text))

    if has_suspicious_url:
        d2_threat_score = 25
    elif has_general_url:
        d2_threat_score = 10
        suspicious_urls.append("일반 외부 웹링크")
    else:
        d2_threat_score = 0

    # ── [D3-1] 긴급성/FOMO 심리 조작 탐지 ──────────────────────────────
    for urg_pat in URGENCY_PATTERNS:
        found_urg = re.findall(urg_pat, raw_text, re.IGNORECASE) or re.findall(urg_pat, norm_text, re.IGNORECASE)
        if found_urg:
            ukw = found_urg[0] if isinstance(found_urg[0], str) else " ".join(found_urg[0])
            urgency_kws.append(ukw)

    urgency_score = 10 if urgency_kws else 0

    # ── [D4] 수신 채널 & 컨텍스트 위험 점수 (최대 15점) ─────────────────
    channel_info = CHANNEL_WEIGHTS.get(channel, CHANNEL_WEIGHTS["기타"])
    # 룰베이스 시그널 또는 URL이 있을 때만 채널 위험 가산
    if d1_rule_score > 0 or d2_threat_score > 0:
        d4_channel_score = channel_info["score"]
    else:
        d4_channel_score = 0

    return {
        "d1_rule_score": d1_rule_score,
        "d2_threat_score": d2_threat_score,
        "urgency_score": urgency_score,
        "d4_channel_score": d4_channel_score,
        "critical_kws": list(set(critical_kws)),
        "minor_kws": list(set(minor_kws)),
        "urgency_kws": list(set(urgency_kws)),
        "suspicious_urls": list(set(suspicious_urls)),
        "category_scores": category_scores,
        "channel_risk_info": channel_info["info"]
    }


def predict(model, text: str, channel: str = "SMS/MMS") -> Dict[str, Any]:
    """
    실제 금융 FDS 4대 차원별 정밀 스코어링 추론
    """
    start_time = time.perf_counter()

    # 1. 딥러닝 감정 분석
    ai_result = model(text)[0]
    sentiment = ai_result["label"]
    confidence = round(float(ai_result["score"]), 4)

    # 2. FDS 다차원 시그널 분석
    sig = _evaluate_fds_signals(text, channel)

    # 3. [D3] AI 언어 & 심리 조작 점수 계산 (최대 20점)
    # 긴급성 점수(최대 10점) + BERT 감정 선동 점수(최대 10점)
    bert_intent_score = 0
    if sig["d1_rule_score"] > 0 or sig["d2_threat_score"] > 0:
        if sentiment == "positive":
            bert_intent_score = int(confidence * 10)
        elif sentiment == "negative":
            bert_intent_score = int(confidence * 10)
        else:
            bert_intent_score = int(confidence * 6)

    d3_intent_score = min(20, sig["urgency_score"] + bert_intent_score)

    # 4. 최종 4대 차원 점수 합산 (최대 100점)
    d1 = sig["d1_rule_score"]
    d2 = sig["d2_threat_score"]
    d3 = d3_intent_score
    d4 = sig["d4_channel_score"]

    raw_total = d1 + d2 + d3 + d4

    # 완전히 정상인 메시지 (시그널 없음)
    if d1 == 0 and d2 == 0 and len(sig["urgency_kws"]) == 0:
        final_risk_score = 5
        scam_type = "안전 / 정상 안내"
        action_guide = "[안전] 정상적인 금융 메시지로 확인되었습니다. 특이 사기 징후가 없습니다."
    else:
        final_risk_score = min(100, max(15, raw_total))
        # 최고 점수 카테고리 결정
        if max(sig["category_scores"].values()) > 0:
            scam_type = max(sig["category_scores"].items(), key=lambda x: x[1])[0]
            action_guide = SCAM_CATEGORIES_RULES[scam_type]["guide"]
        else:
            scam_type = "미확인 출처 의심 링크 / 피싱 의심"
            action_guide = "[주의] 출처가 불분명한 외부 링크가 포함되어 있습니다. 피싱 사이트 접속 및 악성 앱 설치에 각별히 유의하세요."

    # 5. 위험 등급 4단계 판정
    if final_risk_score >= 76:
        risk_level = "고위험 (사기 유력)"
    elif final_risk_score >= 51:
        risk_level = "경고 (사기 의심)"
    elif final_risk_score >= 21:
        risk_level = "주의 (주의 요망)"
    else:
        risk_level = "안전 (정상 메시지)"

    # 5대 사기 카테고리별 정규화된 위험 기여도 (0~100)
    category_distribution = {}
    for cat, raw_sc in sig["category_scores"].items():
        if raw_sc > 0:
            category_distribution[cat] = min(100, raw_sc + d2 + (d3 // 2))
        else:
            category_distribution[cat] = 0

    latency_ms = round((time.perf_counter() - start_time) * 1000.0, 2)

    return {
        "risk_score": final_risk_score,
        "risk_level": risk_level,
        "scam_type": scam_type,
        "score_breakdown": {
            "dimension_1_rule_score": d1,
            "dimension_2_threat_score": d2,
            "dimension_3_intent_score": d3,
            "dimension_4_channel_score": d4
        },
        "category_distribution": category_distribution,
        "detected_signals": {
            "critical_keywords": sig["critical_kws"],
            "minor_keywords": sig["minor_kws"],
            "urgency_keywords": sig["urgency_kws"],
            "suspicious_urls": sig["suspicious_urls"],
            "channel_risk_info": sig["channel_risk_info"]
        },
        "action_guide": action_guide,
        "model_sentiment": sentiment,
        "model_confidence": confidence,
        "latency_ms": latency_ms
    }


Overwriting app/model_service.py


### 4.2 멀티모달 OCR 텍스트 추출 모듈 (`app/ocr_service.py`)
- EasyOCR (한국어/영어) 모델을 통해 캡처 사진 바이트에서 텍스트를 실시간으로 자동 추출합니다.

In [6]:
%%writefile app/ocr_service.py
"""
청년 금융사기 탐지(Youth Financial Scam Guard) 멀티모달 OCR 서비스 모듈
- 의심 문자, 카카오톡, 텔레그램 스크린샷 캡처 이미지에서 텍스트 자동 추출 (Korean & English)
- EasyOCR 기반 경량 CPU/GPU 비동기 지원
"""
import io
import time
from typing import Tuple
from PIL import Image
import numpy as np
import easyocr

_ocr_reader = None


def get_ocr_reader():
    """EasyOCR Reader 인스턴스 싱글톤 로드"""
    global _ocr_reader
    if _ocr_reader is None:
        print("[OCRService] EasyOCR 모델 로딩 시작 (한국어/영어)...")
        _ocr_reader = easyocr.Reader(['ko', 'en'], gpu=False)
        print("[OCRService] EasyOCR 모델 로딩 완료!")
    return _ocr_reader


def extract_text_from_image_bytes(image_bytes: bytes) -> Tuple[str, float]:
    """
    이미지 바이트 데이터를 입력받아 텍스트 추출 및 소요시간 반환
    """
    start_time = time.perf_counter()
    reader = get_ocr_reader()

    # PIL Image 변환 및 RGB 통일
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    image_np = np.array(image)

    # OCR 텍스트 추출
    results = reader.readtext(image_np)

    # 신뢰도 0.2 이상 텍스트 병합
    extracted_lines = [text for (_, text, conf) in results if conf >= 0.2 and len(text.strip()) > 0]
    full_text = " ".join(extracted_lines)

    ocr_latency_ms = round((time.perf_counter() - start_time) * 1000.0, 2)
    return full_text, ocr_latency_ms


Overwriting app/ocr_service.py


### 4.3 FastAPI 비동기 메인 애플리케이션 (`app/main.py`)
- `POST /predict` (텍스트 FDS 추론) & `POST /predict/image` (멀티모달 이미지 OCR FDS 추론)
- `run_in_executor` 스레드 풀 격리 및 `X-API-Key` 보안 의존성 주입

In [7]:
%%writefile app/main.py
"""
청년 대상 금융사기·불법 리딩방 위험 탐지기 (Youth Financial Scam Guard) FastAPI 서버 (멀티모달 OCR 포함)
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor
from contextlib import asynccontextmanager
from fastapi import FastAPI, Depends, HTTPException, status, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from app.auth import verify_api_key
from app.schemas import ScamDetectRequest, ScamDetectResponse, ImageScamDetectResponse, HealthResponse
from app.model_service import load_model, predict
from app.ocr_service import get_ocr_reader, extract_text_from_image_bytes

ml_models = {}
executor = ThreadPoolExecutor(max_workers=4)

@asynccontextmanager
async def lifespan(app: FastAPI):
    print("[Lifespan] Server starting: Loading KR-FinBert-SC and EasyOCR...")
    ml_models["classifier"] = load_model()
    get_ocr_reader()
    print("[Lifespan] All AI models ready!")
    yield
    ml_models.clear()
    executor.shutdown(wait=True)

app = FastAPI(
    title="🛡️ 청년 금융사기·불법 리딩방 위험 탐지 API (FDS & OCR)",
    description="2030 사회초년생 및 청년 대상 불법 금융사기 실시간 탐지, 4대 차원 스코어링 및 스크린샷 OCR 멀티모달 서비스",
    version="1.1.0",
    lifespan=lifespan
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health", response_model=HealthResponse, tags=["시스템 상태"])
async def health_check():
    return HealthResponse(
        status="healthy",
        model_loaded="classifier" in ml_models,
        service_name="Youth Financial Scam Guard (FDS & OCR)",
        version="1.1.0"
    )

@app.post("/predict", response_model=ScamDetectResponse, tags=["금융사기 탐지"])
async def detect_scam(
    payload: ScamDetectRequest,
    user: str = Depends(verify_api_key)
):
    """텍스트 기반 금융사기 FDS 4대 차원 정밀 분석"""
    if "classifier" not in ml_models:
        raise HTTPException(status_code=503, detail="AI 모델이 아직 로드되지 않았습니다.")
    loop = asyncio.get_running_loop()
    try:
        result = await loop.run_in_executor(
            executor,
            predict,
            ml_models["classifier"],
            payload.text,
            payload.channel
        )
        return ScamDetectResponse(
            success=True,
            risk_score=result["risk_score"],
            risk_level=result["risk_level"],
            scam_type=result["scam_type"],
            score_breakdown=result["score_breakdown"],
            category_distribution=result["category_distribution"],
            detected_signals=result["detected_signals"],
            action_guide=result["action_guide"],
            model_sentiment=result["model_sentiment"],
            model_confidence=result["model_confidence"],
            user=user,
            latency_ms=result["latency_ms"]
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"분석 중 오류 발생: {str(e)}")

@app.post("/predict/image", response_model=ImageScamDetectResponse, tags=["멀티모달 이미지 OCR 탐지"])
async def detect_scam_image(
    file: UploadFile = File(..., description="의심 문자, 카카오톡, 텔레그램 캡처 이미지 (PNG, JPG, JPEG, WEBP)"),
    channel: str = Form(default="SMS/MMS", description="수신 채널 (SMS/MMS, Telegram, Instagram DM, OpenChat, KakaoTalk)"),
    user: str = Depends(verify_api_key)
):
    """
    📸 의심 문자/카카오톡 캡처 이미지(스크린샷) 업로드 $\\rightarrow$ OCR 텍스트 자동 추출 $\\rightarrow$ 금융 FDS 정밀 분석
    """
    if "classifier" not in ml_models:
        raise HTTPException(status_code=503, detail="AI 모델이 아직 로드되지 않았습니다.")

    # 이미지 파일 형식 검증
    allowed_types = ["image/png", "image/jpeg", "image/jpg", "image/webp"]
    if file.content_type not in allowed_types:
        raise HTTPException(
            status_code=400,
            detail=f"지원하지 않는 파일 형식입니다 ({file.content_type}). PNG, JPG, JPEG, WEBP 이미지를 업로드하세요."
        )

    image_bytes = await file.read()
    if len(image_bytes) == 0:
        raise HTTPException(status_code=400, detail="빈 이미지 파일입니다.")

    loop = asyncio.get_running_loop()
    try:
        # 1. OCR 텍스트 추출 (스레드 풀 비동기 격리)
        extracted_text, ocr_latency_ms = await loop.run_in_executor(
            executor,
            extract_text_from_image_bytes,
            image_bytes
        )

        if len(extracted_text.strip()) < 3:
            raise HTTPException(
                status_code=422,
                detail="이미지에서 유의미한 텍스트를 감지하지 못했습니다. 글자가 선명한 캡처 사진을 업로드해 주세요."
            )

        # 2. 금융사기 FDS 정밀 분석
        result = await loop.run_in_executor(
            executor,
            predict,
            ml_models["classifier"],
            extracted_text,
            channel
        )

        return ImageScamDetectResponse(
            success=True,
            risk_score=result["risk_score"],
            risk_level=result["risk_level"],
            scam_type=result["scam_type"],
            score_breakdown=result["score_breakdown"],
            category_distribution=result["category_distribution"],
            detected_signals=result["detected_signals"],
            action_guide=result["action_guide"],
            model_sentiment=result["model_sentiment"],
            model_confidence=result["model_confidence"],
            extracted_text=extracted_text,
            ocr_latency_ms=ocr_latency_ms,
            user=user,
            latency_ms=result["latency_ms"]
        )
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"이미지 OCR 분석 중 오류 발생: {str(e)}")


Overwriting app/main.py


### 4.4 백엔드 서버 기동 및 Swagger UI 확인

In [8]:
# ── 백엔드 서버 기동 ────────────────────────────────────────────────
serve_in_thread("app.main:app", port=8000)

⚠️ 포트 8000를 다른 프로세스가 사용 중입니다.


In [9]:
# ── Swagger UI 대화형 테스트 링크 ───────────────────────────────────
from IPython.display import display, HTML
display(HTML('''
<h4>📖 FastAPI 대화형 API 문서 (Swagger UI)</h4>
<a href="http://127.0.0.1:8000/docs" target="_blank" style="font-size:16px; font-weight:bold; color:#2563EB;">
👉 http://127.0.0.1:8000/docs (새 창에서 열기)
</a>
'''))

---

## 5장. Streamlit 멀티모달 인터랙티브 웹 대시보드 구축

### 5.1 프론트엔드 대시보드 (`frontend/app.py`)
- `[탭 1: ✍️ 텍스트 직접 입력]` + `[탭 2: 📸 의심 문자/카톡 캡처 사진 업로드 (OCR)]`
- FDS 4대 차원 분해 카드, 5대 사기 유형 레이더 바, 금감원 1332 신고서 자동 완성기

In [10]:
%%writefile frontend/app.py
"""
청년 대상 금융사기·불법 리딩방 위험 탐지기 (Youth Financial Scam Guard)
Streamlit 인터랙티브 멀티모달 웹 대시보드 (v5 텍스트 + 스크린샷 캡처 OCR 지원)
"""
import streamlit as st
import requests
import io
from PIL import Image

st.set_page_config(
    page_title="청년 금융사기 탐지기 (FDS & OCR)",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── 사이드바 ────────────────────────────────────────────────────────
st.sidebar.title("🔑 보안 인증 & 환경 설정")
api_key = st.sidebar.text_input("X-API-Key 헤더 입력", value="test-key-001", type="password")
server_url = st.sidebar.text_input("FastAPI 서버 주소", value="http://127.0.0.1:8000")

if st.sidebar.button("서버 헬스체크", use_container_width=True):
    try:
        r = requests.get(f"{server_url}/health", timeout=3)
        if r.status_code == 200:
            st.sidebar.success(f"✅ 정상 가동 중 (v{r.json().get('version')})")
            st.sidebar.caption(f"엔진: {r.json().get('service_name')}")
        else:
            st.sidebar.error(f"❌ 상태코드: {r.status_code}")
    except Exception as e:
        st.sidebar.error(f"연결 실패: {e}")

st.sidebar.markdown("---")
st.sidebar.markdown("### 🏦 금융 FDS 4대 차원 배점")
st.sidebar.caption("""
1. **[D1] 시그니처 룰 (40점)**: 불법 키워드
2. **[D2] 위협 인텔리전스 (25점)**: 단축/피싱 링크
3. **[D3] AI 심리 의도 (20점)**: 긴급성/FOMO
4. **[D4] 채널 위험도 (15점)**: 텔레그램/DM 가중치
""")

st.sidebar.markdown("---")
st.sidebar.info("""
**📞 금융사기 긴급 신고처**
- 금융감독원: ☎️ **1332**
- 경찰청 사이버수사국: ☎️ **182 / 112**
- 서민금융진흥원: ☎️ **1397**
""")

# ── 메인 화면 ────────────────────────────────────────────────────────
st.title("🛡️ 청년 금융사기·불법 리딩방 위험 탐지기")
st.markdown("2030 청년을 노리는 금융사기를 **텍스트 직접 입력** 또는 **카톡/문자 캡처 사진 업로드(OCR)**를 통해 실시간으로 다차원 정밀 진단합니다.")

def render_analysis_result(d, input_text, channel, ocr_time=None):
    """분석 결과 공통 렌더링 헬퍼 함수"""
    score = d["risk_score"]
    b = d["score_breakdown"]
    sig = d["detected_signals"]

    # 1. 상단 종합 메트릭
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("🚨 종합 위험 점수", f"{score} / 100")
    m2.metric("🏷️ 위험 등급", d["risk_level"])
    m3.metric("📂 주요 사기 유형", d["scam_type"])
    if ocr_time:
        m4.metric("⏱️ 분석 지연시간", f"{d['latency_ms']} ms (OCR: {ocr_time} ms)")
    else:
        m4.metric("⏱️ 분석 지연시간", f"{d['latency_ms']} ms")

    st.progress(score / 100.0)

    # 2. 금융 FDS 4대 차원 점수 분해 카드
    st.markdown("---")
    st.subheader("📊 금융 FDS 4대 차원별 위험도 분해 (Score Breakdown)")

    f1, f2, f3, f4 = st.columns(4)
    with f1:
        st.info(f"**[D1] 시그니처 룰**\n### `{b['dimension_1_rule_score']}` / 40점\n불법 키워드 매칭")
    with f2:
        st.warning(f"**[D2] 위협 인텔리전스**\n### `{b['dimension_2_threat_score']}` / 25점\n피싱/단축 링크")
    with f3:
        st.error(f"**[D3] AI 언어·심리 의도**\n### `{b['dimension_3_intent_score']}` / 20점\n긴급성/FOMO 선동")
    with f4:
        st.success(f"**[D4] 채널 위험도**\n### `{b['dimension_4_channel_score']}` / 15점\n{channel} 수신 가중치")

    # 3. 5대 사기 유형별 위험 기여도 분포
    st.markdown("---")
    st.subheader("📈 5대 사기 유형별 위험 기여도 분포")
    cat_dist = d["category_distribution"]

    c_cols = st.columns(5)
    for idx, (cat_name, cat_val) in enumerate(cat_dist.items()):
        with c_cols[idx]:
            st.caption(f"**{cat_name}**")
            st.progress(cat_val / 100.0)
            st.write(f"`{cat_val}점`")

    # 4. 탐지된 세부 시그널 및 행동 요령
    st.markdown("---")
    st.subheader("📋 정밀 진단 소견 및 피해 예방 행동 요령")
    if score >= 50:
        st.error(f"🚨 **[금융사기 주의보 발령]** {d['action_guide']}")
    else:
        st.success(f"🟢 **[안전]** {d['action_guide']}")

    st.markdown("##### 🔍 FDS 탐지 시그널 상세 내역")
    s_col1, s_col2 = st.columns(2)
    with s_col1:
        if sig["critical_keywords"]:
            st.write("• **🚨 핵심 범죄 키워드**:", ", ".join(sig["critical_keywords"]))
        if sig["minor_keywords"]:
            st.write("• **⚠️ 보조 의심 키워드**:", ", ".join(sig["minor_keywords"]))
        if sig["urgency_keywords"]:
            st.write("• **⏳ 긴급성/FOMO 압박 단어**:", ", ".join(sig["urgency_keywords"]))
    with s_col2:
        if sig["suspicious_urls"]:
            st.write("• **🌐 피싱/단축 의심 링크**:", ", ".join(sig["suspicious_urls"]))
        st.write("• **📲 채널 컨텍스트**:", sig["channel_risk_info"])

    # 5. 금감원 신고서 양식
    if score >= 50:
        st.markdown("---")
        st.subheader("📑 금융감독원(1332) 간편 신고 서식")
        report_text = f"""[금융사기 의심 제보서]
1. 수신 채널: {channel}
2. 의심 사기 유형: {d['scam_type']}
3. FDS 위험 점수: {score}점 ({d['risk_level']})
   - [D1] 시그니처 룰: {b['dimension_1_rule_score']}점
   - [D2] 위협 링크: {b['dimension_2_threat_score']}점
   - [D3] 심리 조작: {b['dimension_3_intent_score']}점
   - [D4] 채널 위험: {b['dimension_4_channel_score']}점
4. 감지된 단서:
   - 핵심 키워드: {', '.join(sig['critical_keywords'])}
   - 의심 링크: {', '.join(sig['suspicious_urls'])}
5. 메시지 내용:
{input_text}
"""
        st.text_area("신고 복사용 텍스트", report_text, height=130)


# ── 탭 분기 ────────────────────────────────────────────────────────
tab1, tab2 = st.tabs(["✍️ 1. 텍스트 직접 입력", "📸 2. 의심 문자/카톡 캡처 사진 업로드 (OCR)"])

# ── [탭 1] 텍스트 입력 모드 ─────────────────────────────────────────
with tab1:
    SAMPLE_1 = "[WEB 발신] (주)OO커머스 일일 업무 알바 모집. 당일 정산 최소 30만 원 보장, 누구나 즉시 가능! 단시간 고수익 카톡 문의 http://bit.ly/job-scam"
    SAMPLE_2 = "[특별 지원] 급전/신불자 가능! 누구나 당일 최대 500만 원 즉시 입금. 휴대폰 개통 대납 및 소액결제 현금화, 빠른 상담 http://bit.ly/phone-scam"
    SAMPLE_3 = "[국민행복기금] 정부 지원 저금리 대환대출 안내. 기존 고금리 대출 최대 1.5% 대환 가능. 한도 소진 전 무료 상담 신청 http://bit.ly/loan-scam"
    SAMPLE_4 = "[KB국민은행] 고객님의 청년도약계좌 3회차 자동이체가 정상 처리되었습니다. 잔액 1,500,000원"

    if "txt" not in st.session_state:
        st.session_state["txt"] = SAMPLE_1

    st.markdown("##### 💡 빠른 테스트용 실전 의심 메시지 프리셋")
    c1, c2, c3, c4 = st.columns(4)
    with c1:
        if st.button("⚠️ 1. 고수익 알바 사기", key="btn_sample_1", use_container_width=True):
            st.session_state["txt"] = SAMPLE_1
    with c2:
        if st.button("🚨 2. 폰테크/소액결제", key="btn_sample_2", use_container_width=True):
            st.session_state["txt"] = SAMPLE_2
    with c3:
        if st.button("🚨 3. 정부대환대출 사칭", key="btn_sample_3", use_container_width=True):
            st.session_state["txt"] = SAMPLE_3
    with c4:
        if st.button("🟢 4. 정상 은행 알림", key="btn_sample_4", use_container_width=True):
            st.session_state["txt"] = SAMPLE_4

    with st.form("text_scam_form"):
        user_input = st.text_area(
            "분석할 문자/카톡/SNS 텍스트 입력:",
            value=st.session_state["txt"],
            height=110,
            placeholder="수신된 의심 문자나 카카오톡 내용을 여기에 붙여넣으세요..."
        )
        col_a, col_b = st.columns([1, 3])
        with col_a:
            channel_text = st.selectbox("수신 채널", ["SMS/MMS", "Telegram", "Instagram DM", "OpenChat", "KakaoTalk", "기타"], key="ch_text")
        with col_b:
            st.caption("※ 채널별 위험도 + 단축 URL 인스펙터 + AI 금융 BERT 결합")
        submit_text_btn = st.form_submit_button("🔍 텍스트 금융사기 FDS 정밀 분석", use_container_width=True)

    if submit_text_btn:
        if len(user_input.strip()) < 5:
            st.warning("⚠️ 최소 5자 이상 입력하세요.")
        elif not api_key.strip():
            st.error("🔒 사이드바에서 X-API-Key를 입력하세요.")
        else:
            with st.spinner("금융 FDS 4대 차원 및 금융 감정 모델 정밀 분석 중..."):
                try:
                    res = requests.post(
                        f"{server_url}/predict",
                        json={"text": user_input.strip(), "channel": channel_text},
                        headers={"X-API-Key": api_key.strip()},
                        timeout=10
                    )
                    if res.status_code == 200:
                        st.success("✅ FDS 정밀 분석 완료!")
                        render_analysis_result(res.json(), user_input, channel_text)
                    elif res.status_code == 401:
                        st.error("🔒 [인증 실패 401] X-API-Key가 올바르지 않습니다.")
                    elif res.status_code == 422:
                        st.error("⚠️ [입력 검증 오류 422] 텍스트 형식이 맞지 않습니다.")
                    else:
                        st.error(f"❌ [서버 오류 {res.status_code}] {res.text}")
                except Exception as e:
                    st.error(f"🚨 API 접속 실패: {e}")

# ── [탭 2] 이미지 OCR 모드 ─────────────────────────────────────────
with tab2:
    st.markdown("##### 📸 의심스러운 카카오톡 대화방 / 문자 메시지 캡처 사진을 업로드하세요")
    st.caption("AI Vision OCR 엔진이 이미지에서 텍스트를 자동으로 읽어내어 금융사기 위험도를 즉시 분석합니다.")

    uploaded_file = st.file_uploader(
        "캡처 이미지 파일 선택 (PNG, JPG, JPEG, WEBP)",
        type=["png", "jpg", "jpeg", "webp"],
        help="휴대폰 화면 캡처 사진이나 의심 문자 스크린샷을 드래그 앤 드롭하세요."
    )

    col_img_a, col_img_b = st.columns([1, 2])
    with col_img_a:
        channel_img = st.selectbox("수신 채널", ["KakaoTalk", "Telegram", "Instagram DM", "SMS/MMS", "OpenChat", "기타"], key="ch_img")

    if uploaded_file is not None:
        col1, col2 = st.columns([1, 1])
        with col1:
            st.markdown("###### 🖼️ 업로드된 스크린샷 미리보기")
            image = Image.open(uploaded_file)
            st.image(image, use_column_width=True)

        with col2:
            st.markdown("###### 🚀 멀티모달 OCR 분석 실행")
            st.write("선택된 파일명:", uploaded_file.name)
            st.write("파일 크기:", f"{round(len(uploaded_file.getvalue()) / 1024, 1)} KB")

            ocr_submit_btn = st.button("🔍 스크린샷 텍스트 추출 & 사기 분석", use_container_width=True, type="primary")

        if ocr_submit_btn:
            if not api_key.strip():
                st.error("🔒 사이드바에서 X-API-Key를 입력하세요.")
            else:
                with st.spinner("🤖 EasyOCR로 이미지 속 텍스트 추출 및 FDS 금융사기 분석 중..."):
                    try:
                        files = {
                            "file": (uploaded_file.name, uploaded_file.getvalue(), uploaded_file.type)
                        }
                        data = {"channel": channel_img}
                        headers = {"X-API-Key": api_key.strip()}

                        res = requests.post(
                            f"{server_url}/predict/image",
                            files=files,
                            data=data,
                            headers=headers,
                            timeout=25
                        )
                        if res.status_code == 200:
                            d = res.json()
                            st.success("✅ OCR 텍스트 추출 및 FDS 정밀 분석 완료!")

                            st.markdown("---")
                            st.subheader("📝 이미지에서 자동 추출된 텍스트")
                            st.info(d["extracted_text"])

                            render_analysis_result(d, d["extracted_text"], channel_img, ocr_time=d.get("ocr_latency_ms"))
                        elif res.status_code == 401:
                            st.error("🔒 [인증 실패 401] X-API-Key가 올바르지 않습니다.")
                        elif res.status_code == 422:
                            st.error(f"⚠️ [이미지 인식 실패] {res.json().get('detail')}")
                        else:
                            st.error(f"❌ [서버 오류 {res.status_code}] {res.text}")
                    except Exception as e:
                        st.error(f"🚨 API 접속 실패: {e}")


Overwriting frontend/app.py


> ### 📸 [스크린샷 2] Streamlit 웹 대시보드 — 텍스트 직접 입력 및 FDS 4대 차원 진단
>
> ![Streamlit 텍스트 진단 화면](images/02_streamlit_text_demo.png)

> ### 📸 [스크린샷 3] Streamlit 멀티모달 OCR — 캡처 사진 업로드 & 자동 텍스트 추출 및 진단
>
> ![Streamlit 멀티모달 OCR 진단 화면](images/03_streamlit_ocr_demo.png)


### 5.2 프론트엔드 서버 실행 및 웹 UI 대시보드 연동

In [11]:
# ── Streamlit 프론트엔드 구동 ──────────────────────────────────────
run_streamlit("frontend/app.py", port=8501)

♻️  Streamlit이 이미 실행 중입니다: http://127.0.0.1:8501


In [12]:
# ── Streamlit 대시보드 바로가기 링크 ────────────────────────────────
display(HTML('''
<h4>🛡️ 청년 금융사기 탐지기 Streamlit 웹 대시보드</h4>
<a href="http://127.0.0.1:8501" target="_blank" style="font-size:16px; font-weight:bold; color:#10B981;">
👉 http://127.0.0.1:8501 (웹 브라우저에서 열기)
</a>
'''))

---

## 6장. 엔드투엔드(End-to-End) 서비스 통합 검증

### 6.1 4대 보안 및 데이터 유효성 시나리오 테스트

In [13]:
import requests

API_URL = "http://127.0.0.1:8000"

print("=== [테스트 1] API Key 누락 요청 (401 확인) ===")
r1 = requests.post(f"{API_URL}/predict", json={"text": "원금 500% 보장 VIP 리딩방"})
print(f"상태코드: {r1.status_code} | 응답: {r1.json()}")
assert r1.status_code == 401

print("\n=== [테스트 2] 잘못된 API Key 요청 (401 확인) ===")
r2 = requests.post(f"{API_URL}/predict", json={"text": "원금 500% 보장 VIP 리딩방"}, headers={"X-API-Key": "wrong-key"})
print(f"상태코드: {r2.status_code} | 응답: {r2.json()}")
assert r2.status_code == 401

print("\n=== [테스트 3] 입력 길이 부족 (422 유효성 검증) ===")
r3 = requests.post(f"{API_URL}/predict", json={"text": "단문"}, headers={"X-API-Key": "test-key-001"})
print(f"상태코드: {r3.status_code}")
assert r3.status_code == 422

print("\n=== [테스트 4] 정상 사기 탐지 요청 (200 OK) ===")
r4 = requests.post(f"{API_URL}/predict", json={"text": "[긴급] 이번 주 500% 폭등 확정 VIP 리딩방 선착순 10명 무료 입장! 세력 매집주 원금 보장 100% 정보방 http://bit.ly/vip-scam", "channel": "Telegram"}, headers={"X-API-Key": "test-key-001"})
print(f"상태코드: {r4.status_code} | 위험점수: {r4.json().get('risk_score')}점 | 유형: {r4.json().get('scam_type')}")
assert r4.status_code == 200

print("\n🎉 4대 보안/검증 시나리오 테스트를 성공적으로 통과했습니다!")

=== [테스트 1] API Key 누락 요청 (401 확인) ===
상태코드: 401 | 응답: {'detail': 'API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.'}

=== [테스트 2] 잘못된 API Key 요청 (401 확인) ===
상태코드: 401 | 응답: {'detail': '유효하지 않은 API Key입니다.'}

=== [테스트 3] 입력 길이 부족 (422 유효성 검증) ===
상태코드: 422

=== [테스트 4] 정상 사기 탐지 요청 (200 OK) ===
상태코드: 200 | 위험점수: 95점 | 유형: 불법 주식/코인 리딩방

🎉 4대 보안/검증 시나리오 테스트를 성공적으로 통과했습니다!


> ### 📸 [스크린샷 4] FastAPI 대화형 API 명세서 (Swagger UI Docs)
>
> ![FastAPI Swagger UI API Docs](images/04_fastapi_swagger_docs.png)


### 6.2 5종 사기 유형별 종합 분석 벤치마크 테스트

In [14]:
# ── 금융 FDS 4대 차원 및 5종 사기 유형별 종합 분석 벤치마크 ─────────────────
test_cases = [
    ("1. 불법 리딩방", "[긴급] 이번 주 500% 폭등 확정 VIP 리딩방 선착순 10명 무료 입장! 세력 매집주 원금 보장 100% 정보방 http://bit.ly/vip-scam", "Telegram"),
    ("2. SNS 고수익 알바", "[WEB 발신] (주)OO커머스 일일 업무 알바 모집. 당일 정산 최소 30만 원 보장, 누구나 즉시 가능! 단시간 고수익 카톡 문의 http://bit.ly/job-scam", "Instagram DM"),
    ("3. 폰테크/대리입금", "[특별 지원] 급전/신불자 가능! 누구나 당일 최대 500만 원 즉시 입금. 휴대폰 개통 대납 및 소액결제 현금화, 빠른 상담 http://bit.ly/phone-scam", "SMS/MMS"),
    ("4. 대환대출/기관사칭", "[국민행복기금] 정부 지원 저금리 대환대출 안내. 기존 고금리 대출 최대 1.5% 대환 가능. 한도 소진 전 무료 상담 신청 http://bit.ly/loan-scam", "SMS/MMS"),
    ("5. 정상 은행 알림", "[KB국민은행] 고객님의 청년도약계좌 3회차 자동이체가 정상 처리되었습니다. 잔액 1,500,000원", "KakaoTalk")
]

print(f"{'유형':<20} | {'위험점수':<8} | {'D1(룰)':<7} | {'D2(링크)':<8} | {'D3(심리)':<8} | {'D4(채널)':<8} | {'위험등급':<14}")
print("-" * 95)

for label, text, ch in test_cases:
    resp = requests.post(
        f"{API_URL}/predict",
        json={"text": text, "channel": ch},
        headers={"X-API-Key": "test-key-001"}
    )
    if resp.status_code == 200:
        d = resp.json()
        b = d["score_breakdown"]
        print(f"{label:<20} | {str(d['risk_score'])+'점':<8} | {str(b['dimension_1_rule_score'])+'점':<7} | {str(b['dimension_2_threat_score'])+'점':<8} | {str(b['dimension_3_intent_score'])+'점':<8} | {str(b['dimension_4_channel_score'])+'점':<8} | {d['risk_level']:<14}")
        print(f"  └─ 감지단서: {d['detected_signals']['critical_keywords']} | 링크: {d['detected_signals']['suspicious_urls']}")
    else:
        print(f"{label:<20} | 에러: {resp.status_code}")

print("-" * 95)
print("✅ 금융 FDS 4대 차원 종합 벤치마크 테스트 완료!")


유형                   | 위험점수     | D1(룰)   | D2(링크)   | D3(심리)   | D4(채널)   | 위험등급          
-----------------------------------------------------------------------------------------------
1. 불법 리딩방            | 95점      | 40점     | 25점      | 15점      | 15점      | 고위험 (사기 유력)   
  └─ 감지단서: ['VIP 리딩방', '원금 보장', '500% 폭등'] | 링크: ['bit.ly/vip-scam']
2. SNS 고수익 알바        | 82점      | 40점     | 25점      | 5점       | 12점      | 고위험 (사기 유력)   
  └─ 감지단서: ['단시간 고수익', '당일 정산'] | 링크: ['bit.ly/job-scam']
3. 폰테크/대리입금          | 78점      | 40점     | 25점      | 5점       | 8점       | 고위험 (사기 유력)   
  └─ 감지단서: ['휴대폰 개통 대납', '소액결제 현금화'] | 링크: ['bit.ly/phone-scam']
4. 대환대출/기관사칭         | 88점      | 40점     | 25점      | 15점      | 8점       | 고위험 (사기 유력)   
  └─ 감지단서: ['국민행복기금', '저금리 대환대출'] | 링크: ['bit.ly/loan-scam']
5. 정상 은행 알림          | 5점       | 0점      | 0점       | 0점       | 0점       | 안전 (정상 메시지)   
  └─ 감지단서: [] | 링크: []
----------------------------------------------------------------------------

### 6.3 멀티모달 이미지 OCR 사기 탐지 테스트 (`POST /predict/image`)

In [15]:
# ── 이미지 파일 업로드 OCR 탐지 테스트 ──────────────────────────────
image_path = "data/sample_scam_screenshot.png"
with open(image_path, "rb") as f:
    files = {"file": ("sample.png", f, "image/png")}
    data = {"channel": "KakaoTalk"}
    headers = {"X-API-Key": "test-key-001"}
    
    resp = requests.post(f"{API_URL}/predict/image", files=files, data=data, headers=headers)

if resp.status_code == 200:
    d = resp.json()
    print(f"✅ 이미지 OCR 분석 성공! (지연시간: {d['latency_ms']}ms | OCR: {d['ocr_latency_ms']}ms)")
    print(f"📝 추출된 텍스트: {d['extracted_text']}")
    print(f"🚨 위험 점수: {d['risk_score']}점 | 등급: {d['risk_level']} | 유형: {d['scam_type']}")
    print(f"📊 FDS 점수 분해: {d['score_breakdown']}")
else:
    print(f"❌ 실패: {resp.status_code} | {resp.text}")


✅ 이미지 OCR 분석 성공! (지연시간: 76.01ms | OCR: 1884.68ms)
📝 추출된 텍스트: [카카오록] 7 긴급 금움 투자 안내 [긴급] 이번 주 5009 폭등 확정 VIP 리팅방 선작순 10명 무료 입장! 세력 매집주 원금 1009 보장! 단타 극비 정보방 림크 클렉 단축 굉크: http:/lbitlylvip-scam * 오늘 마감 전 서둘러 입장하지 않으면 참여가 제한티니다
🚨 위험 점수: 60점 | 등급: 경고 (사기 의심) | 유형: 불법 주식/코인 리딩방
📊 FDS 점수 분해: {'dimension_1_rule_score': 40, 'dimension_2_threat_score': 0, 'dimension_3_intent_score': 15, 'dimension_4_channel_score': 5}


---

## 7장. 8일간의 여정 총정리 & MLOps 확장 로드맵 (Conclusion)

### 7.1 Day 1~8 서빙 기술 발전 지도
```text
Day 1: 모델 직렬화 & 환경 구성 (PyTorch / Joblib) -> "모델을 저장하고 불러온다"
Day 2: FastAPI & Pydantic 스키마 정의          -> "모델을 REST API로 감싸고 타입을 검증한다"
Day 3: 비동기 처리 & run_in_executor           -> "CPU 무거운 추론 시 서버 블로킹을 방지한다"
Day 4: Streamlit 웹 대시보드 구축              -> "비개발자도 쓸 수 있는 인터랙티브 UI를 붙인다"
Day 5: [프로젝트 1] 주택 가격 예측 서비스       -> "정형 데이터 전처리/추론 파이프라인을 완성한다"
Day 6: API Key 인증 & 이미지 미디어 서빙        -> "보안 인증과 비정형 데이터 처리를 다룬다"
Day 7: [프로젝트 2] 트랜스포머 챗봇 & RAG 서빙  -> "생성형 모델과 벡터 DB 검색을 서빙한다"
Day 8: [자율 프로젝트] 청년 금융사기 탐지기     -> "멀티모달 OCR + 4대 차원 FDS 엔터프라이즈 AI 안전망 구축"
```

---

### 7.2 Day 8 최종 5대 체크포인트 질의응답

* **Q1. 본인의 프로젝트에서 Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?**
  * **답변**: 
    1. **텍스트 유효성 검증**: `min_length=5`로 무의미한 공백·초단문을 방어하고, `max_length=3000`으로 서버 메모리를 고갈시키는 비정상 대용량 페이로드를 사전 차단(`422 Unprocessable Entity`)합니다.
    2. **멀티모달 이미지 검증**: 이미지 업로드 시 비정상 파일 확장자나 텍스트가 없는 빈 파일을 검증하여 유효한 이미지(PNG, JPG, WEBP)만 파이프라인에 유입되도록 통제합니다.
    3. **정밀 응답 스키마 직렬화**: `ScoreBreakdown`(D1~D4) 및 `DetectedSignals` 등 10개 이상의 FDS 분석 메트릭이 누락 없이 엄격한 데이터 타입으로 클라이언트에 전달되도록 보장합니다.

* **Q2. Depends(verify_api_key)를 제거하면 어떤 위험이 있습니까?**
  * **답변**: 
    1. **DoS 공격 및 리소스 고갈**: 본 서비스는 BERT 순전파 및 EasyOCR 비전 연산 등 고비용 CPU/GPU 연산을 수반합니다. 인증이 제거되면 악의적 봇이 무차별적으로 요청을 보내 서버를 다운시킬 수 있습니다.
    2. **비즈니스 보안 붕괴**: 금융 FDS API는 금융기관·핀테크 앱의 핵심 보안 자산이므로, API Key를 통해 인가된 클라이언트만 식별하고 향후 엔드포인트별 사용량 제한(Rate Limiting) 및 과금 체계를 적용하기 위해 필수적입니다.

* **Q3. run_in_executor를 사용한 이유는 무엇입니까?**
  * **답변**: 
    1. **이벤트 루프 블로킹 방지**: PyTorch 기반 BERT 추론과 EasyOCR 텍스트 추출 연산은 동기(Synchronous) CPU 바운드 작업입니다.
    2. 이를 FastAPI의 단일 비동기 이벤트 루프에서 직접 실행하면 연산이 끝날 때까지 서버 전체가 멈춰 다른 사용자의 요청이나 헬스체크(`/health`)를 처리하지 못합니다. 따라서 `loop.run_in_executor(ThreadPoolExecutor)`로 무거운 연산을 별도 워커 스레드 풀에 격리하여 고성능 동시성을 확보했습니다.

* **Q4. Day 1~8 중 가장 많이 참고한 Day는 어디였습니까? 왜?**
  * **답변**: **Day 2(Pydantic 스키마 설계), Day 3(비동기 스레드 풀 격리), Day 6(API Key 헤더 보안 및 Streamlit 통합)**을 가장 유기적으로 결합했습니다. 단순 모델 코드가 아닌 실제 상용 서비스 레벨의 **[보안 → 유효성 검증 → 비동기 격리 → 사용자 UI]** 4단계 엔지니어링 파이프라인을 완성하는 핵심 토대가 되었기 때문입니다.

* **Q5. 이 서비스를 실제로 상용 배포하려면 추가로 무엇이 필요합니까?**
  * **답변**:
    1. **추론 가속화**: BERT 및 EasyOCR 모델을 ONNX Runtime 또는 TensorRT(INT8 양자화)로 변환하여 P99 지연시간을 50ms 이내로 단축.
    2. **캐싱 인프라**: Redis를 도입하여 동일한 단축 URL이나 중복 이미지 해시(SHA-256)에 대해 모델 재추론 없이 즉각 캐시 응답(0ms).
    3. **클라우드 MLOps & 오토스케일링**: Docker 컨테이너화 및 Kubernetes(K8s) HPA 기반 오토스케일링, Prometheus/Grafana 지연시간 실시간 모니터링 구축.

---

### 7.3 KPT 회고 (Keep, Problem, Try) & 차세대 AI 모델 로드맵

* **Keep (프로젝트에서 지속 유지할 훌륭한 점)**:
  * **설명 가능한 금융 FDS 아키텍처 구축**: AI 딥러닝 감정 분석의 블랙박스 한계를 극복하기 위해, 실제 시중은행 FDS 구조인 `4대 차원(D1 시그니처 40점 + D2 위협 링크 25점 + D3 심리 의도 20점 + D4 수신 채널 15점)`으로 점수를 분해하여 높은 신뢰도와 설명 가능성(XAI)을 달성한 점.
  * **멀티모달 OCR 확장**: 텍스트 입력의 번거로움을 없애고 청년들이 카카오톡·문자 캡처 사진만 올리면 즉시 분석되는 실용적인 UI/UX를 구현한 점.
  * **실질적 사회적 가치 창출**: 단순 점수 출력을 넘어 금감원 1332 간편 제보서 자동 생성 등 실전 피해 예방 프로세스를 완성한 점.

* **Problem (겪었던 기술적 한계 및 문제점)**:
  * **사전학습 도메인 갭(Domain Gap)**: 뉴스 호악재 감정 분석용 `KR-FinBert-SC`가 사기 범죄 문자를 `중립(99.9%)`으로 오판하여, 딥러닝 단독으로는 사기 탐지가 불가능했던 한계.
  * **키워드 부재 시 미탐 위험**: 사기범이 등록된 키워드를 교묘하게 피해 갈 경우(Zero-day 사기), 룰베이스(D1)가 0점이 되어 위험도가 과소평가될 수 있는 구조적 한계.

* **Try (차세대 AI 모델 발전 로드맵 — Next Generation Roadmap)**:
  1. **[단기 로드맵] 한국어 문맥 유사도 모델(`jhgan/ko-sbert-multitask`) 도입**:
     - 사기 분류기가 아닌 **Sentence-BERT 문맥 임베딩**을 적용하여, 키워드가 완전히 달라도 사기 템플릿과의 의미론적 코사인 유사도(Semantic Cosine Similarity)를 측정해 키워드 없는 신종 사기까지 100% 포착.
  2. **[중기 로드맵] 금융사기 전용 경량 sLLM (Few-Shot Reasoning) 파인튜닝**:
     - `Qwen2.5-1.5B` 또는 `Llama-3.2-1B` 모델에 금감원 사기 피해 판례 데이터를 LoRA로 파인튜닝하여, 복잡한 신종 사기의 숨은 의도와 인과관계를 설명하는 생성형 FDS 엔진 구축.
  3. **[인프라 로드맵] 실시간 위협 인텔리전스 & 컨테이너 MLOps**:
     - 금감원 실시간 보이스피싱 DB 연동, 단축 URL 실시간 IP 언팩커(Unpacker), Docker/K8s 기반 고가용성 무중단 서빙 파이프라인 구축.
